# 03 — BERTopic topic modeling

Discovers 80 topics on the corpus, then reduces to 15 interpretable clusters. Each review gets a topic_id, including the BERTopic outlier label `-1` for reviews that do not cluster cleanly. The outlier flag becomes a feature in the XGBoost classifier downstream.

**This notebook is slow on the full corpus.** Default uses a 5,000-review stratified sample for tractable runtime. Set `SAMPLE_SIZE = None` to run on all 48,147 reviews.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

from src.data_loader import load_reviews
from src.config import (
    OUTPUTS_DIR, BERTOPIC_EMBEDDING_MODEL,
    BERTOPIC_NUM_TOPICS_RAW, BERTOPIC_NUM_TOPICS_REDUCED,
)

SAMPLE_SIZE = 5000   # set to None for full 48,147

In [ ]:
df = load_reviews(clean=True)
if SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
texts = df['text'].fillna('').tolist()
print(f'Modeling {len(texts):,} reviews')

## Embed with sentence-transformers, then cluster

Embedding model: `all-MiniLM-L6-v2`, 384 dimensions. Reduce dimensionality with UMAP to 5 components, then cluster with HDBSCAN. BERTopic stitches all of this together.

In [ ]:
embedding_model = SentenceTransformer(BERTOPIC_EMBEDDING_MODEL)
embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=64)

umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0,
                  metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean',
                        cluster_selection_method='eom', prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=BERTOPIC_NUM_TOPICS_RAW,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(texts, embeddings)
print(f'Discovered {len(set(topics))} topic ids (including outlier -1)')

## Reduce to 15 interpretable topics

In [ ]:
topic_model.reduce_topics(texts, nr_topics=BERTOPIC_NUM_TOPICS_REDUCED)
topics = topic_model.topics_
print(f'Reduced to {len(set(topics))} topics (including outlier -1)')
print()
print(topic_model.get_topic_info().head(20))

## Save topic assignments

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
out = df[['review_id']].copy()
out['topic_id'] = topics
out['is_topic_outlier'] = (out['topic_id'] == -1).astype(int)
out_path = OUTPUTS_DIR / 'topic_assignments.csv'
out.to_csv(out_path, index=False)
print(f'Saved {len(out):,} rows to {out_path.relative_to(out_path.parent.parent)}')
print(f'Outlier rate: {out["is_topic_outlier"].mean():.1%}')

## Notes

- The reduction from 80 to 15 is motivated by interpretability. Eighty topics is   too many for a human to label, fifteen is at the boundary of legibility.
- The outlier rate is meaningful, reviews that BERTopic cannot place are often the   shortest or the most generic. Both are weak signals for authenticity in either   direction.
- BERTopic embeddings are not portable across machines. The pipeline regenerates   them on each run. For CI speed we use the 5,000-review sample by default.